In [3]:
# 03_dictionary_management.ipynb
# Dynamic dictionary/key management with forward secrecy

import numpy as np
import hashlib
import hmac
import time
import json
import os
from datetime import datetime, timedelta

# --- 1. DICTIONARY MANAGER CLASS ---

class DynamicDictionaryManager:
    def __init__(self, manufacturer_key=None, device_id=None):
        """Initialize dictionary manager with manufacturer key and device ID"""
        self.manufacturer_key = manufacturer_key or os.urandom(32)
        self.device_id = device_id or self._generate_device_id()
        self.dictionaries = {}  # In-memory storage for demo
        self.refresh_interval = 3600  # 1 hour
        
    def _generate_device_id(self):
        """Generate unique device identifier"""
        # In practice, this would use hardware features
        import platform
        import uuid
        system_info = f"{platform.system()}_{platform.machine()}_{uuid.getnode()}"
        return hashlib.sha256(system_info.encode()).hexdigest()[:16]
    
    def generate_shared_dictionary(self, peer_device_id, session_id, size=256):
        """Generate shared dictionary for two devices"""
        # Create deterministic seed from both device IDs
        combined_seed = f"{min(self.device_id, peer_device_id)}_{max(self.device_id, peer_device_id)}_{session_id}"
        seed_hash = hashlib.pbkdf2_hmac('sha256', 
                                       combined_seed.encode(), 
                                       self.manufacturer_key, 
                                       100000)
        
        # Set numpy seed for deterministic generation
        np.random.seed(int.from_bytes(seed_hash[:4], 'big'))
        
        # Generate dictionary entries
        dictionary = {}
        for i in range(size):
            key = np.random.randint(0, 2**16, dtype=np.uint16)
            value = np.random.bytes(8)  # 8-byte values
            dictionary[key] = value
        
        # Store with metadata
        self.dictionaries[session_id] = {
            'dictionary': dictionary,
            'created_at': datetime.now(),
            'peer_device': peer_device_id,
            'expires_at': datetime.now() + timedelta(seconds=self.refresh_interval)
        }
        
        return dictionary

# --- 2. INITIALIZE DICTIONARY MANAGER ---

# Device setup
device_id = "device_alice_001"
peer_device_id = "device_bob_002"
manufacturer_key = os.urandom(32)  # In practice, this is pre-provisioned

dict_manager = DynamicDictionaryManager(manufacturer_key, device_id)
print(f"Device ID: {dict_manager.device_id}")
print(f"Manufacturer Key: {manufacturer_key.hex()}")

# --- 3. GENERATE SHARED DICTIONARY ---

session_id = f"session_{int(time.time())}"
shared_dict = dict_manager.generate_shared_dictionary(peer_device_id, session_id, size=64)

print(f"\nGenerated shared dictionary for session: {session_id}")
print(f"Dictionary size: {len(shared_dict)} entries")
print("Sample entries:")
for i, (key, value) in enumerate(list(shared_dict.items())[:5]):
    print(f"  {key}: {value.hex()}")

# --- 4. DICTIONARY LOOKUP FUNCTION ---

def dictionary_lookup(dict_manager, session_id, lookup_key):
    """Look up value in shared dictionary"""
    if session_id in dict_manager.dictionaries:
        dictionary = dict_manager.dictionaries[session_id]['dictionary']
        return dictionary.get(lookup_key, None)
    return None

# Test lookup
test_key = list(shared_dict.keys())[0]
test_value = dictionary_lookup(dict_manager, session_id, test_key)
print(f"\nDictionary lookup test:")
print(f"Key: {test_key}")
print(f"Value: {test_value.hex()}")
print(f"Lookup successful: {test_value == shared_dict[test_key]}")

# --- 5. FORWARD SECRECY / KEY ROTATION ---

def refresh_dictionary(dict_manager, session_id, peer_device_id):
    """Refresh dictionary with forward secrecy"""
    old_session_id = session_id
    new_session_id = f"{session_id}_refresh_{int(time.time())}"
    
    # Generate new dictionary
    new_dict = dict_manager.generate_shared_dictionary(peer_device_id, new_session_id)
    
    # Remove old dictionary (forward secrecy)
    if old_session_id in dict_manager.dictionaries:
        del dict_manager.dictionaries[old_session_id]
        print(f"Old dictionary for session {old_session_id} deleted")
    
    print(f"New dictionary generated for session: {new_session_id}")
    return new_dict, new_session_id

# Test key rotation
print("\n--- KEY ROTATION TEST ---")
new_dict, new_session_id = refresh_dictionary(dict_manager, session_id, peer_device_id)
print(f"New dictionary size: {len(new_dict)}")
print(f"Old session accessible: {session_id in dict_manager.dictionaries}")
print(f"New session accessible: {new_session_id in dict_manager.dictionaries}")

# --- 6. SECURE DICTIONARY EXCHANGE PROTOCOL ---

def secure_dictionary_handshake(dict_manager, peer_device_id, session_id):
    """Simulate secure dictionary exchange protocol"""
    # Generate challenge
    challenge = os.urandom(16)
    
    # Create response using manufacturer key
    response_data = f"{dict_manager.device_id}_{peer_device_id}_{challenge.hex()}"
    response = hmac.new(dict_manager.manufacturer_key, 
                       response_data.encode(), 
                       hashlib.sha256).digest()
    
    # Generate shared dictionary
    dictionary = dict_manager.generate_shared_dictionary(peer_device_id, session_id)
    
    return {
        'status': 'success',
        'challenge': challenge.hex(),
        'response': response.hex(),
        'dictionary_size': len(dictionary)
    }

# Test secure handshake
handshake_session = f"handshake_session_{int(time.time())}"
handshake_result = secure_dictionary_handshake(dict_manager, peer_device_id, handshake_session)
print(f"\n--- SECURE HANDSHAKE TEST ---")
print(f"Status: {handshake_result['status']}")
print(f"Challenge: {handshake_result['challenge']}")
print(f"Response: {handshake_result['response'][:32]}...")
print(f"Dictionary size: {handshake_result['dictionary_size']}")

# --- 7. INTEGRATION WITH TENSOR ENCRYPTION ---

def dictionary_enhanced_encryption(message, tensor_key, dictionary, session_id):
    """Encrypt using both tensor key and dictionary values"""
    # Get dictionary enhancement values
    dict_keys = list(dictionary.keys())[:16]  # Use first 16 keys
    dict_values = [dictionary[k] for k in dict_keys]
    
    # Create enhancement tensor from dictionary
    enhancement_array = np.array([int.from_bytes(v[:1], 'big') for v in dict_values], dtype=np.uint8)
    enhancement_tensor = enhancement_array.reshape(4, 4)
    
    # Combine tensor key with dictionary enhancement
    combined_key = np.bitwise_xor(tensor_key, enhancement_tensor)
    
    # Encrypt message
    message_array = np.array([ord(c) for c in message], dtype=np.uint8).reshape(4, 4)
    encrypted = np.bitwise_xor(message_array, combined_key)
    
    return encrypted, combined_key

# Test dictionary-enhanced encryption
test_message = "DictEncryption01"
test_tensor_key = np.random.randint(0, 256, (4, 4), dtype=np.uint8)
dictionary = dict_manager.dictionaries[new_session_id]['dictionary']

encrypted_msg, enhanced_key = dictionary_enhanced_encryption(
    test_message, test_tensor_key, dictionary, new_session_id
)

print(f"\n--- DICTIONARY-ENHANCED ENCRYPTION TEST ---")
print("Original message:", test_message)
print("Encrypted array:")
print(encrypted_msg)

# Decrypt to verify
decrypted_array = np.bitwise_xor(encrypted_msg, enhanced_key)
decrypted_message = ''.join([chr(x) for x in decrypted_array.flatten()])
print("Decrypted message:", decrypted_message)
print(f"Encryption/decryption successful: {decrypted_message == test_message}")


Device ID: device_alice_001
Manufacturer Key: 3f34f73485aadd988808aa09c3f4993f3e590bd3874021bf35bfdf9412008b32

Generated shared dictionary for session: session_1759174317
Dictionary size: 64 entries
Sample entries:
  54726: 4ecc64c714f96d43
  41782: 21931d6939787a32
  62255: 5693fbd4f650cdda
  50229: df692735831bcd98
  53843: c8b4b7caec85f3ec

Dictionary lookup test:
Key: 54726
Value: 4ecc64c714f96d43
Lookup successful: True

--- KEY ROTATION TEST ---
Old dictionary for session session_1759174317 deleted
New dictionary generated for session: session_1759174317_refresh_1759174317
New dictionary size: 256
Old session accessible: False
New session accessible: True

--- SECURE HANDSHAKE TEST ---
Status: success
Challenge: b0d193006acb91117dd88b8ac28f59c4
Response: 530b85510de3658ab852407ed3230ac6...
Dictionary size: 256

--- DICTIONARY-ENHANCED ENCRYPTION TEST ---
Original message: DictEncryption01
Encrypted array:
[[  8   9 203  17]
 [180  14 186  57]
 [234 229  17 142]
 [106 100 168 114